In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"D:\DS115-VIS\.venv\Project1_Predictive-Maintenance\ai4i_fused.csv")
df.head()


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,PWF,OSF,RNF,date,avg_temp_c,min_temp_c,max_temp_c,precipitation_mm,avg_wind_speed_kmh,avg_sea_level_pres_hpa
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4


In [4]:
# Clean trailing and leading spaces from column names
df.columns = df.columns.str.strip()

In [5]:
# 2. Sort data for time-series compatibility
# Chronological order is mandatory for accurate rolling calculations
df = df.sort_values(by=['date', 'UDI']).reset_index(drop=True)

In [6]:
# 3. Select Target Telemetry Features
# These are the main continuous sensor and weather columns from image
sensor_features = [
    'Air tempe', 'Process te', 'Rotationa', 'Torque', 'Tool wear',
    'avg_temp', 'min_temp', 'max_temp', 'precipitat', 'avg_wind', 'avg_sea_level_pres_hpa'
]

In [7]:
# Define the rolling window size (Operational window size - e.g., 5 logs)
window_size = 5
df.head(5)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,PWF,OSF,RNF,date,avg_temp_c,min_temp_c,max_temp_c,precipitation_mm,avg_wind_speed_kmh,avg_sea_level_pres_hpa
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4


In [8]:
print("Starting feature engineering...")


Starting feature engineering...


In [9]:
# 4. Engineer Rolling Mean, Standard Deviation, and Variance
# Perform calculations inside a new DataFrame
engineered_features = pd.DataFrame(index=df.index)


In [10]:
for col in sensor_features:
    # Check if the column exists in the dataset
    if col in df.columns:
        # Rolling Mean
        engineered_features[f'{col}_roll_mean'] = df[col].rolling(window=window_size).mean()
        
        # Rolling Standard Deviation
        engineered_features[f'{col}_roll_std'] = df[col].rolling(window=window_size).std()
        
        # Rolling Variance
        engineered_features[f'{col}_roll_var'] = df[col].rolling(window=window_size).var()
df

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,PWF,OSF,RNF,date,avg_temp_c,min_temp_c,max_temp_c,precipitation_mm,avg_wind_speed_kmh,avg_sea_level_pres_hpa
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,M24855,M,298.8,308.4,1604,29.5,14,0,0,...,0,0,0,2021-02-20,25.1,23.0,28.0,0.0,16.0,1013.1
9996,9997,H39410,H,298.9,308.4,1632,31.8,17,0,0,...,0,0,0,2021-02-20,25.1,23.0,28.0,0.0,16.0,1013.1
9997,9998,M24857,M,299.0,308.6,1645,33.4,22,0,0,...,0,0,0,2021-02-20,25.1,23.0,28.0,0.0,16.0,1013.1
9998,9999,H39412,H,299.0,308.7,1408,48.5,25,0,0,...,0,0,0,2021-02-20,25.1,23.0,28.0,0.0,16.0,1013.1


In [11]:
# 5. Combine with original metadata and targets
# Includes UDI, Product ID, Machine failure, and specific failure modes
meta_cols = ['UDI', 'Product ID', 'Type', 'date', 'Machine f', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
# Filter out names to match what is present in the dataset
meta_cols = [c for c in meta_cols if c in df.columns]

final_df = pd.concat([df[meta_cols], engineered_features], axis=1)


In [ ]:
# 6. Drop NaN rows generated by the rolling window warmup buffer
final_df.dropna(inplace=True)
df.head(10)


In [12]:
# 7. Verify the output structure and save the file
print(f"Processed dataset shape: {final_df.shape}")
print(final_df.head())

# Save the final engineered dataset
final_df.to_csv("engineered_iot_weather_dataset.csv", index=False)
print("File successfully saved as 'engineered_iot_weather_dataset.csv'!")


Processed dataset shape: (10000, 12)
   UDI Product ID Type        date  TWF  HDF  PWF  OSF  RNF  \
0    1     M14860    M  2020-01-01    0    0    0    0    0   
1    2     L47181    L  2020-01-01    0    0    0    0    0   
2    3     L47182    L  2020-01-01    0    0    0    0    0   
3    4     L47183    L  2020-01-01    0    0    0    0    0   
4    5     L47184    L  2020-01-01    0    0    0    0    0   

   avg_sea_level_pres_hpa_roll_mean  avg_sea_level_pres_hpa_roll_std  \
0                               NaN                              NaN   
1                               NaN                              NaN   
2                               NaN                              NaN   
3                               NaN                              NaN   
4                            1014.4                              0.0   

   avg_sea_level_pres_hpa_roll_var  
0                              NaN  
1                              NaN  
2                              NaN  
3  

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [14]:
# 1. Define original sensor features & target variable
# Using only the original features as requested for the baseline
base_features = ['Air tempe', 'Process te', 'Rotationa', 'Torque', 'Tool wear']
target = 'Machine f'  
# Assuming 'Machine f' stands for Machine Failure 
# Filter features that exist in df to avoid any KeyErrors
base_features = [col for col in base_features if col in df.columns]